# मायक्रोसॉफ्ट एजंट फ्रेमवर्कसह ह्युमन-इन-द-लूप वर्कफ्लो

## 🎯 शिकण्याचे उद्दिष्टे

या नोटबुकमध्ये, तुम्ही मायक्रोसॉफ्ट एजंट फ्रेमवर्कच्या `RequestInfoExecutor` वापरून **ह्युमन-इन-द-लूप** वर्कफ्लो कसे अंमलात आणायचे हे शिकाल. हा सामर्थ्यशाली पॅटर्न AI वर्कफ्लो थांबवून मानवी इनपुट गोळा करण्यास परवानगी देतो, ज्यामुळे तुमचे एजंट संवादात्मक होतात आणि मानवी नियंत्रण महत्त्वाच्या निर्णयांवर असते.

## 🔄 ह्युमन-इन-द-लूप म्हणजे काय?

**ह्युमन-इन-द-लूप (HITL)** हा एक डिझाईन पॅटर्न आहे ज्यात AI एजंट कार्यवाही थांबवतात आणि पुढे जाण्यापूर्वी मानवी इनपुट मागतात. हे महत्त्वाचे आहे:

- ✅ **महत्त्वाचे निर्णय** - महत्त्वाचे निर्णय घेण्याअगोदर मानवी मान्यता मिळवा
- ✅ **अस्पष्ट परिस्थिती** - AI अनिश्चित असताना मानव्यांना स्पष्टता द्या
- ✅ **वापरकर्त्याच्या प्राधान्यक्रम** - अनेक पर्यायांमध्ये वापरकर्त्यांना निवड करण्यासाठी विचारा
- ✅ **सामंजस्य आणि सुरक्षितता** - नियमन केलेल्या क्रियांसाठी मानवी देखरेखीची खात्री करा
- ✅ **संवादात्मक अनुभव** - वापरकर्ता इनपुटला प्रतिसाद देणारे संभाषणात्मक एजंट तयार करा

## 🏗️ मायक्रोसॉफ्ट एजंट फ्रेमवर्कमध्ये ते कसे कार्य करते

या फ्रेमवर्कमध्ये HITL साठी तीन मुख्य घटक आहेत:

1. **`RequestInfoExecutor`** - एक विशेष एक्झिक्युटर जो वर्कफ्लो थांबवतो आणि `RequestInfoEvent` दर्शवितो
2. **`RequestInfoMessage`** - मानवाद्वारे पाठवलेल्या टाइप केलेल्या विनंती पेलोडसाठी बेस क्लास
3. **`RequestResponse`** - `request_id` वापरून मानवी प्रतिसादांना मूळ विनंत्यांशी जोडतो

**वर्कफ्लो पॅटर्न:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 आमचे उदाहरण: वापरकर्ता पुष्टीसह हॉटेल बुकिंग

आपण सशर्त वर्कफ्लोवर आधारित, पर्यायी ठिकाणे सुचवण्याअगोदर मानवी पुष्टी जोडणार आहोत:

1. वापरकर्ता ठिकाणासाठी विनंती करतो (उदा. "पॅरिस")
2. `availability_agent` तपासतो की खोल्या उपलब्ध आहेत का
3. **जर खोली नसतील** → `confirmation_agent` विचारतो "आपण पर्यायी पाहू इच्छिता का?"
4. वर्कफ्लो `RequestInfoExecutor` वापरून **थांबतो**
5. **मानव प्रतिसाद देतो** "होय" किंवा "नाही" कन्सोल इनपुटद्वारे
6. `decision_manager` प्रतिसादानुसार मार्गदर्शन करतो:
   - **होय** → पर्यायी ठिकाणे दाखवा
   - **नाही** → बुकिंग विनंती रद्द करा
7. अंतिम निकाल दर्शवा

हे दर्शविते की वापरकर्त्यांना एजंटच्या सूचना नियंत्रित करण्यास कशी संधी दिली जाऊ शकते!

---

चला सुरुवात करूया! 🚀


## पाऊल 1: आवश्यक लायब्ररी आयात करा

आम्ही मानक एजंट फ्रेमवर्क घटकांव्यतिरिक्त **मानव-इन-द-लूप विशिष्ट वर्ग** आयात करतो:
- `RequestInfoExecutor` - एक्जिक्युटर जे मानवी इनपुटसाठी कार्यप्रवाह थांबवते
- `RequestInfoEvent` - मानवी इनपुट मागितल्यावर प्रसारित होणारा इव्हेंट
- `RequestInfoMessage` - टाइप केलेल्या विनंती पेलोडसाठी मूलभूत वर्ग
- `RequestResponse` - मानवी प्रतिसादांची विनंत्यांसह जुळवणी करते
- `WorkflowOutputEvent` - कार्यप्रवाह आउटपुट शोधण्यासाठी इव्हेंट


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## टप्पा 2: संरचित आउटपुटसाठी Pydantic मॉडेल्स परिभाषित करा

हे मॉडेल्स एजंट परत करणारे **स्कीमा** परिभाषित करतात. आम्ही अटीच्या कार्यप्रवाहातील सर्व मॉडेल्स ठेवतो आणि यामध्ये जोडतो:

**मानव-इन-दी-लूपसाठी नवीन:**
- `HumanFeedbackRequest` - `RequestInfoMessage` चा उपवर्ग जो मानवांना पाठवले जाणारे विनंती पेलोड परिभाषित करतो
  - यात `prompt` (विचारायचा प्रश्न) आणि `destination` (अप्राप्य शहराबद्दल संदर्भ) समाविष्ट आहे


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## टप्पा 3: हॉटेल बुकिंग साधन तयार करा

अटीवर आधारित कार्यप्रवाहातीलच तोच साधन - गंतव्यस्थानी खोल्या उपलब्ध आहेत का ती तपासते.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## टप्पा 4: राउटिंगसाठी स्थिती फंक्शन्स परिभाषित करा

आपल्या मानव-इन-द-लूप वर्कफ्लोसाठी **चार स्थिती फंक्शन्स** आवश्यक आहेत:

**स्थितीय वर्कफ्लो मधून:**
1. `has_availability_condition` - जेव्हा हॉटेल उपलब्ध असतात तेव्हा राउट करते
2. `no_availability_condition` - जेव्हा हॉटेल उपलब्ध नसतात तेव्हा राउट करते

**मनुष्य-इन-द-लूपसाठी नवीन:**
3. `user_wants_alternatives_condition` - जेव्हा वापरकर्ता "होय" म्हणतो पर्यायांसाठी तेव्हा राउट करते
4. `user_declines_alternatives_condition` - जेव्हा वापरकर्ता "नाही" म्हणतो पर्यायांसाठी तेव्हा राउट करते


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## पायरी 5: डिसिजन मॅनेजर एक्झिक्युटर तयार करा

हा **मानव-इन-द-लूप पॅटर्नचा मुख्य भाग** आहे! `DecisionManager` हा एक सानुकूलित `Executor` आहे जो:

1. `RequestResponse` वस्तूंच्या माध्यमातून माणसाचा अभिप्राय प्राप्त करतो
2. वापरकर्त्याचा निर्णय (होय/नाही) प्रक्रियेत आणतो
3. योग्य एजंट्सना संदेश पाठवून कार्यप्रवाह नियंत्रित करतो

मुख्य वैशिष्ट्ये:
- `@handler` डेकोरेटर वापरून पद्धती कार्यप्रवाहाच्या टप्प्यांप्रमाणे उघड करतो
- `RequestResponse[HumanFeedbackRequest, str]` प्राप्त करतो ज्यात मूळ विनंती आणि वापरकर्त्याचा उत्तर असतो
- साधे "होय" किंवा "नाही" संदेश तयार करतो जे आपल्या सशर्त फंक्शन्सना ट्रिगर करतात


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## पायरी 6: कस्टम डिस्प्ले एक्झिक्युटर तयार करा

कंडिशनल वर्कफ्लोमधील समान डिस्प्ले एक्झिक्युटर - वर्कफ्लो आउटपुट म्हणून अंतिम निकाल प्रदान करतो.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## टप्पा 7: पर्यावरण चल लोड करा

LLM क्लायंट (Microsoft Foundry / Azure OpenAI) कॉन्फिगर करा.


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## टप्पा 8: AI एजंट आणि एक्झिक्युटर्स तयार करा

आम्ही **सहा वर्कफ्लो घटक** तयार करतो:

**एजंट्स (AgentExecutor मध्ये गुंडाळलेले):**
1. **availability_agent** - साधन वापरून हॉटेलची उपलब्धता तपासतो
2. **confirmation_agent** - 🆕 मानवी पुष्टी विनंती तयार करतो
3. **alternative_agent** - पर्यायी शहरे सुचवतो (जेव्हा वापरकर्ता होकार देतो)
4. **booking_agent** - बुकिंगस प्रोत्साहन देतो (जेव्हा खोल्या उपलब्ध असतात)
5. **cancellation_agent** - 🆕 रद्दीकरण संदेश हाताळतो (जेव्हा वापरकर्ता नाही म्हणतो)

**विशेष एक्झिक्युटर्स:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` जो मानवी इनपुटसाठी वर्कफ्लो थांबवतो
7. **decision_manager** - 🆕 सानुकूल एक्झिक्युटर जो मानवी प्रतिसादानुसार मार्गक्रमण करतो (ज्याची आधीच व्याख्या केली आहे)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## टप्पा 9: मानवी सहभाग असलेला वर्कफ्लो तयार करा

आता आपण **सशर्त मार्गदर्शन** सह मानवी सहभाग असलेल्या मार्गासह वर्कफ्लो ग्राफ तयार करतो:

**वर्कफ्लो संरचना:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**महत्त्वाच्या काठा:**
- `availability_agent → confirmation_agent` (जेव्हा खोली उपलब्ध नसतील)
- `confirmation_agent → prepare_human_request` (ट्रान्सफॉर्म प्रकार)
- `prepare_human_request → request_info_executor` (मानवासाठी थांबा)
- `request_info_executor → decision_manager` (नेहमी - RequestResponse प्रदान करते)
- `decision_manager → alternative_agent` (जेव्हा वापरकर्ता "हो" म्हणतो)
- `decision_manager → cancellation_agent` (जेव्हा वापरकर्ता "नाही" म्हणतो)
- `availability_agent → booking_agent` (जेव्हा खोली उपलब्ध असतील)
- सर्व मार्ग `display_result` येथे संपतात


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## टप्पा 10: टेस्ट केस 1 चालवा - शहर उपलब्धता शिवाय (पॅरिससाठी मानवी पुष्टीसह)

हा चाचणी **मानव-इन-थी-लूप संपूर्ण चक्र** दर्शवितो:

1. पॅरिसमधील हॉटेलची विनंती करा
2. availability_agent तपासतो → खोल्या नाहीत
3. confirmation_agent मानवी-दृष्टीने प्रश्न तयार करतो
4. request_info_executor **वर्कफ्लो थांबवतो** आणि `RequestInfoEvent` उत्सर्जित करतो
5. **अर्ज कार्यक्रम इव्हेंट ओळखतो आणि कन्सोलमध्ये वापरकर्त्यास प्रॉम्प्ट करतो**
6. वापरकर्ता "होय" किंवा "नाही" टाइप करतो
7. अर्ज प्रतिसाद `send_responses_streaming()` मार्फत पाठवतो
8. decision_manager प्रतिसादाच्या आधारावर मार्गदर्शन करतो
9. अंतिम निकाल प्रदर्शित केला जातो

**महत्त्वाचा पॅटर्न:**
- पहिल्या पुनरावृत्ती साठी `workflow.run_stream()` वापरा
- पुढील पुनरावृत्त्यांसाठी `workflow.send_responses_streaming(pending_responses)` वापरा
- मानवी इनपुट आवश्यक तेव्हा ओळखण्यासाठी `RequestInfoEvent` ऐका
- अंतिम निकाल पकडण्यासाठी `WorkflowOutputEvent` ऐका


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## टप्पा 11: चाचणी प्रकरण 2 चालवा - शहर उपलब्धतेसह (स्टॉकहोम - मानवी इनपुट आवश्यक नाही)

ही चाचणी **थेट मार्ग** दर्शवते जेव्हा खोल्या उपलब्ध असतात:

1. स्टॉकहोममधील हॉटेलची विनंती करा
2. availability_agent तपासणी → खोल्या उपलब्ध आहेत ✅
3. booking_agent बुकिंग सुचवते
4. display_result पुष्टीकरण दर्शवते
5. **कोणताही मानवी इनपुट आवश्यक नाही!**

जेव्हा खोल्या उपलब्ध असतात तेव्हा कार्यप्रवाह मानवी-इन-द-लूप मार्ग पूर्णतः वगळतो.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## मुख्य मुद्दे आणि मानव-इन-द-लूप सर्वोत्तम पद्धती

### ✅ आपण काय शिकलात:

#### 1. **RequestInfoExecutor पॅटर्न**
Microsoft Agent Framework मधील मानव-इन-द-लूप पॅटर्नमध्ये तीन मुख्य घटक वापरले जातात:
- `RequestInfoExecutor` - वर्कफ्लो थांबवतो आणि इव्हेंट्स प्रसारित करतो
- `RequestInfoMessage` - टाइप केलेल्या विनंती पेलोडसाठी बेस क्लास (याचा उपवर्ग करा!)
- `RequestResponse` - मानवी उत्तरांना मूळ विनंत्यांशी संबंधित करतो

**महत्वाचे समज:**
- `RequestInfoExecutor` स्वतः इनपुट गोळा करत नाही - ते फक्त वर्कफ्लो थांबवते
- आपल्या अनुप्रयोगाच्या कोडने `RequestInfoEvent` ऐकणे आणि इनपुट गोळा करणे आवश्यक आहे
- वापरकर्त्याच्या उत्तरासाठी `request_id` मॅपिंगसह `send_responses_streaming()` कॉल करणे आवश्यक आहे

#### 2. **स्ट्रीमिंग अंमलबजावणी पॅटर्न**
```python
# प्रथम पुनरावृत्ती
stream = workflow.run_stream(initial_request)

# नंतरची पुनरावृत्ती (माणसाच्या इनपुटनंतर)
stream = workflow.send_responses_streaming(pending_responses)

# नेहमी घटना प्रक्रिया करा
events = [event async for event in stream]
```

#### 3. **इव्हेंट-चालित आर्किटेक्चर**
वर्कफ्लो नियंत्रित करण्यासाठी विशिष्ट इव्हेंट्स ऐका:
- `RequestInfoEvent` - मानवी इनपुट आवश्यक (वर्कफ्लो थांबलेले)
- `WorkflowOutputEvent` - अंतिम निकाल उपलब्ध (वर्कफ्लो पूर्ण)
- `WorkflowStatusEvent` - स्थिती बदल (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, इ.)

#### 4. **@handler सह सानुकूल Executors**
`DecisionManager` कसे तयार करायचे हे दर्शवितो:
- `@handler` डेकोरेटर वापरून वर्कफ्लो स्टेप्ससाठी मेथड्स उघड करणे
- टाइप केलेले संदेश प्राप्त करणे (उदा., `RequestResponse[HumanFeedbackRequest, str]`)
- संदेश इतर एक्सेक्युटर्सकडे पाठवून वर्कफ्लो मार्गदर्शन करणे
- `WorkflowContext` मार्फत संदर्भ प्राप्त करणे

#### 5. **मानवी निर्णयांसह सशर्त रूटिंग**
आपण मानवी प्रतिसादांचे मूल्यांकन करणारे सशर्त फंक्शन्स तयार करू शकता:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 प्रत्यक्ष जीवनातील अनुप्रयोग:

1. **मंजुरी वर्कफ्लोज**
   - खर्च अहवाल प्रक्रिया करण्याअगोदर व्यवस्थापकाची मंजुरी मिळवा
   - स्वयंचलित ईमेल पाठवण्याअगोदर मानवी पुनरावलोकन आवश्यक आहे
   - उच्च-मूल्य व्यवहार अंमलात आणण्याअगोदर पुष्टी करा

2. **सामग्री मानकेकरण**
   - संशयास्पद सामग्री मानवी पुनरावलोकनासाठी चिन्हांकित करा
   - किनाऱ्या प्रकरणांवर अंतिम निर्णय करण्यासाठी मॉडरेटरना विचारा
   - जेव्हा AI विश्वास कमी असेल तेव्हा मानवाकडे वाढवा

3. **ग्राहक सेवा**
   - AI ला नियमित प्रश्न स्वयंचलितरीत्या हाताळू द्या
   - जटिल समस्या मानव एजंटकडे वाढवा
   - ग्राहकाला विचारा की ते मानवाशी बोलू इच्छितात का

4. **डेटा प्रक्रिया**
   - अस्पष्ट डेटा प्रविष्ट्या मानवाकडे सोडविण्यास विचारणे
   - अस्पष्ट दस्तऐवजावर AI व्याख्या पुष्टी करणे
   - वापरकर्त्यांना अनेक वैध व्याख्यांमधून निवडण्याची परवानगी द्या

5. **सुरक्षित-गंभीर प्रणाली**
   - अपरिवर्तनीय क्रियांपूर्वी मानवी पुष्टी आवश्यक आहे
   - संवेदनशील डेटावर प्रवेश करण्याअगोदर मंजुरी मिळवा
   - नियमन केलेल्या उद्योगांमध्ये निर्णयांची पुष्टी करा (आरोग्य, वित्त)

6. **परस्पर संवादित एजंट्स**
   - संवादात्मक बॉट तयार करा जे पुढील प्रश्न विचारतात
   - वापरकर्त्यांना गुंतागुंतीच्या प्रक्रियांमधून मार्गदर्शन करणारे विजार्ड तयार करा
   - मानवी सहकार्याने टप्प्याटप्प्याने एजंट डिझाइन करा

### 🔄 तुलना: मानव-इन-द-लूप सह व शिवाय

| वैशिष्ट्य | सशर्त वर्कफ्लो | मानव-इन-द-लूप वर्कफ्लो |
|---------|---------------------|---------------------------|
| **अंमलबजावणी** | एकल `workflow.run()` | `run_stream()` + `send_responses_streaming()` सह लूप |
| **वापरकर्ता इनपुट** | नाही (पूर्णपणे स्वयंचलित) | `input()` किंवा UI द्वारे इंटरऍक्टिव्ह प्रॉम्प्ट्स |
| **घटक** | एजंट्स + एक्सेक्युटर | + RequestInfoExecutor + DecisionManager |
| **इव्हेंट्स** | AgentExecutorResponse फक्त | RequestInfoEvent, WorkflowOutputEvent, इ. |
| **थांबवणे** | थांबवणे नाही | RequestInfoExecutor येथे वर्कफ्लो थांबतो |
| **मानवी नियंत्रण** | मानवी नियंत्रण नाही | मानवी महत्त्वपूर्ण निर्णय घेतात |
| **वापर केस** | ऑटोमेशन | सहकार्य आणि देखरेख |

### 🚀 प्रगत पॅटर्न:

#### अनेक मानवी निर्णय बिंदू
आपण एका वर्कफ्लोमध्ये अनेक `RequestInfoExecutor` नोड्स असू शकतात:
```python
.add_edge(agent1, request_info_1)  # पहिला मानवी निर्णय
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # दुसरा मानवी निर्णय
.add_edge(decision_manager_2, final_agent)
```

#### टाइमआउट हँडलिंग
मानवी प्रतिसादांसाठी टाइमआउट लागू करा:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # सुरक्षित पर्यायाला डीफॉल्ट करा
```

#### समृद्ध UI एकीकरण
`input()` ऐवजी, वेब UI, Slack, Teams इ. सोबत एकत्र करा:
```python
if isinstance(event, RequestInfoEvent):
    # वापरकर्त्याच्या प्राधान्य दिलेल्या चॅनेलवर सूचना पाठवा
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### सशर्त मानव-इन-द-लूप
फक्त विशिष्ट परिस्थितींमध्ये मानवी इनपुट मागा:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # केवळ आत्मविश्वास कमी असेल किंवा मूल्य जास्त असेल तरच मानवाकडे मार्गदर्शित करा
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ सर्वोत्तम पद्धती:

1. **RequestInfoMessage सदैव उपवर्ग करा**
   - टाइप सुरक्षा आणि मान्यता प्रदान करते
   - UI रेंडरिंगसाठी समृद्ध संदर्भ सक्षम करते
   - प्रत्येक विनंती प्रकाराचा हेतू स्पष्ट करतो

2. **वर्णनात्मक प्रॉम्प्ट्स वापरा**
   - आपण काय विचारत आहात याबद्दल संदर्भ जोडा
   - प्रत्येक निवडीचे परिणाम स्पष्ट करा
   - प्रश्न सोपे आणि स्पष्ट ठेवा

3. **अपेक्षित नसलेले इनपुट हाताळा**
   - वापरकर्ता प्रतिसादांची मान्यता करा
   - अवैध इनपुटसाठी डीफॉल्ट्स द्या
   - स्पष्ट त्रुटी संदेश द्या

4. **Request IDs ट्रॅक करा**
   - request_id आणि प्रतिसादांमधील संबंधितता वापरा
   - स्थिती मॅन्युअली हाताळण्याचा प्रयत्न करू नका

5. **नॉन-ब्लॉकिंग डिझाइन करा**
   - इनपुटसाठी वाट पाहत त्रेस ब्लॉक करू नका
   - संपूर्णपणे असिंक्रोनस पॅटर्न वापरा
   - एकाधिक वर्कफ्लो उदाहरणांना समर्थन द्या

### 📚 संबंधित संकल्पना:

- **एजंट मिडलवेअर** - एजंट कॉल्स इंटरसेप्ट करा (मागील नोटबुक)
- **वर्कफ्लो स्टेट मॅनेजमेंट** - रन दरम्यान वर्कफ्लो स्थिती जतन करा
- **मल्टी-एजंट सहकार्य** - मानव-इन-द-लूप एजंट टीम्सबरोबर एकत्र करा
- **इव्हेंट-चालित आर्किटेक्चर** - इव्हेंटसह प्रतिक्रियाशील प्रणाली तयार करा

---

### 🎓 अभिनंदन!

आपण Microsoft Agent Framework सोबत मानव-इन-द-लूप वर्कफ्लोमध्ये पारंगत झालात! आता आपण कशी करू शकता हे माहित आहे:
- ✅ मानवी इनपुट गोळा करण्यासाठी वर्कफ्लो थांबवा
- ✅ RequestInfoExecutor आणि RequestInfoMessage वापरा
- ✅ इव्हेंट्ससह स्ट्रीमिंग अंमलबजावणी हाताळा
- ✅ @handler सह सानुकूल एक्सेक्युटर तयार करा
- ✅ मानवी निर्णयांवर आधारित वर्कफ्लो मार्गदर्शन करा
- ✅ मानवांसह संवादात्मक AI एजंट तयार करा जे सहकार्य करतात

**हा विश्वासार्ह, नियंत्रित AI प्रणाली तयार करण्यासाठी एक महत्त्वाचा पॅटर्न आहे!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
